# Chapter 3: Pagination Patterns

Pagination is one of the most important concepts in scraping. Without it you collect only the first page.
This notebook covers all major pagination patterns with live examples.

**Patterns covered:**
1. **Page-number offset** - `?page=1`, `?page=2` (most common)
2. **Numeric offset** - `?start=0`, `?start=20` (Craigslist, eBay style)
3. **Cursor / token-based** - next page token from API response
4. **Last-page detection** - stop conditions
5. **Infinite scroll** (intro - Playwright chapter goes deep)

In [8]:
import httpx
from bs4 import BeautifulSoup
import time

def fetch(url: str) -> BeautifulSoup:
    resp = httpx.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "lxml")

## 3.1 Pattern 1: Page-Number Offset

The URL contains a `?page=N` parameter. This is the most common pattern on modern sites.
The ShopSphere and JobHive demo sites both use this pattern.

In [9]:
# ShopSphere SSR: ?page=1, ?page=2, ...
BASE = "http://localhost:8001"
LINK_SELECTOR = "a.product-link"
MAX_PAGES = 10

all_links = []

for page in range(1, MAX_PAGES + 1):
    url = f"{BASE}/products?page={page}"
    soup = fetch(url)
    links = soup.select(LINK_SELECTOR)
    
    if not links:
        print(f"Page {page}: 0 links - stopping (stop condition: no_results)")
        break
    
    print(f"Page {page}: {len(links)} products")
    all_links.extend([
        BASE + el.get("href", "") if el.get("href", "").startswith("/") else el.get("href", "")
        for el in links
    ])
    time.sleep(0.1)  # polite delay

print(f"\nTotal products found: {len(all_links)}")

Page 1: 10 products
Page 2: 10 products
Page 3: 10 products
Page 4: 10 products
Page 5: 10 products
Page 6: 0 links - stopping (stop condition: no_results)

Total products found: 50


## 3.2 Pattern 2: Numeric Offset

Instead of page numbers, some sites use an item offset: `?start=0&count=20`, `?start=20&count=20`.
The API endpoint of the demo sites accepts `?page=` but many real APIs use offset-based pagination.

In [10]:
# Simulate offset-based pagination using the CSR API endpoint
# /api/products?page=N is our page-based API
# We translate offsets: offset=0 -> page=1, offset=10 -> page=2

PER_PAGE = 10
MAX_OFFSET = 50
api_results = []

for offset in range(0, MAX_OFFSET, PER_PAGE):
    page = offset // PER_PAGE + 1
    resp = httpx.get(f"http://localhost:8002/api/products?page={page}")
    data = resp.json()
    
    products = data.get("products", [])
    if not products:
        print(f"Offset {offset}: empty - stopping")
        break
    
    print(f"Offset {offset:>3} -> Page {page}: {len(products)} products (total: {data['total_items']})")
    api_results.extend(products)

print(f"\nFetched {len(api_results)} products via offset pagination")

Offset   0 -> Page 1: 10 products (total: 50)
Offset  10 -> Page 2: 10 products (total: 50)
Offset  20 -> Page 3: 10 products (total: 50)
Offset  30 -> Page 4: 10 products (total: 50)
Offset  40 -> Page 5: 10 products (total: 50)

Fetched 50 products via offset pagination


## 3.3 Stop Conditions

Knowing when to stop is as important as knowing how to paginate. Common stop conditions:

| Condition | When to use |
|-----------|-------------|
| `no_results` | Stop when the link selector returns 0 elements |
| `last_page_text` | Stop when a string like "No more results" appears |
| `max_pages` | Hard limit regardless of content |
| `api_has_next` | API returns `has_next: false` |
| `http_404` | Next page URL returns 404 |

In [11]:
def scrape_with_stop_conditions(
    base_url: str,
    link_selector: str,
    max_pages: int = 20,
    last_page_text: str = "",
) -> list:
    """
    Generic paginator with multiple stop conditions.
    """
    all_urls = []
    
    for page in range(1, max_pages + 1):
        url = f"{base_url}?page={page}"
        resp = httpx.get(url, headers={"User-Agent": "Mozilla/5.0"})
        html = resp.text
        
        # Stop condition: HTTP error
        if resp.status_code >= 400:
            print(f"  Stop: HTTP {resp.status_code} at page {page}")
            break
        
        # Stop condition: last_page_text found
        if last_page_text and last_page_text.lower() in html.lower():
            print(f"  Stop: last_page_text found at page {page}")
            break
        
        # Stop condition: no_results
        soup = BeautifulSoup(html, "lxml")
        links = soup.select(link_selector)
        if not links:
            print(f"  Stop: no links at page {page}")
            break
        
        all_urls.extend([BASE + el.get("href", "") for el in links])
        print(f"  Page {page}: {len(links)} items")
        time.sleep(0.05)
    
    return all_urls

# ShopSphere - stops at page 6 (only 5 pages exist)
print("Scraping ShopSphere:")
urls = scrape_with_stop_conditions("http://localhost:8001/products", "a.product-link")
print(f"Total: {len(urls)}")

print("\nScraping JobHive:")
job_urls = scrape_with_stop_conditions("http://localhost:8003/jobs", "a.job-link")
print(f"Total: {len(job_urls)}")

Scraping ShopSphere:
  Page 1: 10 items
  Page 2: 10 items
  Page 3: 10 items
  Page 4: 10 items
  Page 5: 10 items
  Stop: no links at page 6
Total: 50

Scraping JobHive:
  Page 1: 10 items
  Page 2: 10 items
  Page 3: 10 items
  Page 4: 10 items
  Stop: no links at page 5
Total: 40


## 3.4 Pattern 3: "Next" Link Traversal

Some sites embed a 'Next Page' link in the HTML rather than using a predictable URL pattern.
You follow the link until there is no more 'next' to follow.

In [12]:
def follow_next_links(
    start_url: str,
    item_selector: str,
    next_selector: str = "a.pagination-btn:last-child",
    base: str = "",
    max_pages: int = 20,
) -> list:
    """
    Follow 'next' links until none exist.
    This pattern is agnostic to URL structure.
    """
    items = []
    current_url = start_url
    seen_urls = set()
    
    for page_num in range(1, max_pages + 1):
        if current_url in seen_urls:
            print(f"  Stop: revisiting {current_url}")
            break
        seen_urls.add(current_url)
        
        soup = fetch(current_url)
        found = soup.select(item_selector)
        items.extend(found)
        print(f"  Page {page_num} ({current_url}): {len(found)} items")
        
        # Find the 'Next' link
        next_el = soup.select_one(next_selector)
        if not next_el:
            print(f"  Stop: no next link")
            break
        
        href = next_el.get("href", "")
        if not href or href == current_url:
            break
        
        current_url = (base + href) if href.startswith("/") else href
    
    return items

# Note: Our demo sites use explicit Next/Previous links with ?page=N URLs
# The pagination-btn with 'Next' text is the last one when has_next is True
print("Following next links on ShopSphere:")
# For demo: get all pagination buttons and pick the one labeled 'Next'
soup = fetch("http://localhost:8001/products")
pagn = soup.select("nav.pagination a.pagination-btn")
print(f"Pagination buttons found: {[b.get_text(strip=True) for b in pagn]}")

Following next links on ShopSphere:
Pagination buttons found: ['1', '2', '3', '4', '5', 'Next']


## 3.5 Pattern 4: API Cursor Pagination

Modern REST APIs often use cursor-based pagination instead of page numbers.
Each response contains a cursor token for the next page.
This pattern is common in Twitter/X API, GitHub API, Stripe API.

In [13]:
# Simulated cursor pagination - the demo CSR API uses page numbers
# but here we simulate cursor logic by converting page numbers to encoded cursors

import base64

def encode_cursor(page: int) -> str:
    """Simulate a cursor token (base64 encoded page number)."""
    return base64.b64encode(f"page:{page}".encode()).decode()

def decode_cursor(cursor: str) -> int:
    decoded = base64.b64decode(cursor.encode()).decode()
    return int(decoded.split(":")[1])

def paginate_with_cursor(api_url: str, per_page: int = 10) -> list:
    """
    Cursor-based pagination pattern:
    - Request returns data + next_cursor
    - Use next_cursor in next request
    - Stop when next_cursor is None/empty
    """
    all_items = []
    cursor = encode_cursor(1)  # Start cursor
    
    while cursor:
        page = decode_cursor(cursor)
        resp = httpx.get(f"{api_url}?page={page}")
        data = resp.json()
        
        items = data.get("products") or data.get("jobs") or []
        if not items:
            break
        
        all_items.extend(items)
        print(f"Cursor page {page}: {len(items)} items (total so far: {len(all_items)})")
        
        # Generate next cursor (real APIs return this in the response)
        has_next = data.get("has_next", False)
        cursor = encode_cursor(page + 1) if has_next else None
    
    return all_items

print("Cursor-based pagination on ShopSphere API:")
products = paginate_with_cursor("http://localhost:8002/api/products")
print(f"\nTotal products: {len(products)}")
print(f"First: {products[0]['name']}")
print(f"Last: {products[-1]['name']}")

Cursor-based pagination on ShopSphere API:
Cursor page 1: 10 items (total so far: 10)
Cursor page 2: 10 items (total so far: 20)
Cursor page 3: 10 items (total so far: 30)
Cursor page 4: 10 items (total so far: 40)
Cursor page 5: 10 items (total so far: 50)

Total products: 50
First: MacBook Pro 14"
Last: Razer DeathAdder V3 HyperSpeed


## 3.6 The bylgja Config Approach

Instead of writing pagination code from scratch each time, the JSON config approach captures the pagination pattern declaratively.

In [14]:
import json

# The pagination section of a scraper config is a complete description
# of how to generate all page URLs

pagination_config = {
    # Page-number pagination (most common)
    "page_based": {
        "url_template": "https://example.com/products?page={n}",
        "pagination": {"start": 1, "step": 1, "max_pages": 50, "stop_condition": "no_results"}
    },
    
    # Offset-based pagination
    "offset_based": {
        "url_template": "https://example.com/items?start={n}&count=20",
        "pagination": {"start": 0, "step": 20, "max_pages": 100, "stop_condition": "no_results"}
    },
    
    # Last-page text detection
    "last_page_text": {
        "url_template": "https://example.com/search?p={n}",
        "pagination": {
            "start": 1, "step": 1, "max_pages": 200,
            "stop_condition": "last_page_text",
            "last_page_text": "No more results"
        }
    }
}

def generate_urls_from_config(source: dict) -> list:
    """Reproduce the bylgja pagination URL generator."""
    template = source["url_template"]
    p = source["pagination"]
    start, step, max_pages = p["start"], p["step"], p["max_pages"]
    return [template.replace("{n}", str(start + i * step)) for i in range(max_pages)]

for name, cfg in pagination_config.items():
    urls = generate_urls_from_config(cfg)
    print(f"{name}: first 3 URLs = {urls[:3]}")

page_based: first 3 URLs = ['https://example.com/products?page=1', 'https://example.com/products?page=2', 'https://example.com/products?page=3']
offset_based: first 3 URLs = ['https://example.com/items?start=0&count=20', 'https://example.com/items?start=20&count=20', 'https://example.com/items?start=40&count=20']
last_page_text: first 3 URLs = ['https://example.com/search?p=1', 'https://example.com/search?p=2', 'https://example.com/search?p=3']


## 3.7 Summary: Choosing a Pagination Strategy

| Site type | URL pattern | Config approach |
|-----------|-------------|------------------|
| Most marketplaces | `?page=N` | `start=1, step=1` |
| eBay-style | `?_pgn=N` | Same, different param |
| Craigslist | `?s=N` (offset) | `start=0, step=120` |
| Old forums | `/page/N/` in path | URL template with `{n}` in path |
| Infinite scroll | No URL change | Playwright + scroll automation |
| REST APIs | Cursor token | Custom agent logic |